Export die to GDS (fractured to <=4000 vertices>)

In [1]:
import re
import gdsfactory as gf
import klayout.db as kdb
from amf.chp import PDK

gf.clear_cache()  
PDK.activate()

In [2]:
from myamf.Main_v2 import Main_v2

c = Main_v2()
# c.plot()    

/workspace/.venv/lib/python3.12/site-packages/gdsfactory/components/bends/bend_euler.py:112: UserWarning: {'width': 1.0} ignored for cross_section 'xs_aef91a05'
  x = gf.get_cross_section(cross_section, width=width or x.width)
/workspace/.venv/lib/python3.12/site-packages/gdsfactory/components/bends/bend_euler.py:112: UserWarning: {'width': 10.0} ignored for cross_section 'metal_routing'
  x = gf.get_cross_section(cross_section, width=width or x.width)
/workspace/.venv/lib/python3.12/site-packages/gdsfactory/components/bends/bend_euler.py:112: UserWarning: {'width': 0.5} ignored for cross_section 'strip'
  x = gf.get_cross_section(cross_section, width=width or x.width)
/workspace/.venv/lib/python3.12/site-packages/gdsfactory/components/bends/bend_euler.py:112: UserWarning: {'width': 1.0} ignored for cross_section 'nitride'
  x = gf.get_cross_section(cross_section, width=width or x.width)
/workspace/.venv/lib/python3.12/site-packages/gdsfactory/components/bends/bend_euler.py:112: UserWa

2026-07-06 19:04:53.254 | ERROR    | kfactory.kcell:dup:781 - When duplicating a Cell, avoid giving the duplicate the same name, as this can cause naming conflicts and may render the GDS/OASIS file unwritable. If you're using a @cell function, ensure that the function has a different name than the one being called.


/workspace/.venv/lib/python3.12/site-packages/gdsfactory/components/bends/bend_euler.py:112: UserWarning: {'width': 0.5} ignored for cross_section 'xs_e2c03727'
  x = gf.get_cross_section(cross_section, width=width or x.width)


[strip_MZI] upper: 122.832 µm | lower: 132.135 µm | ΔL: 9.303 µm (target 9.303 µm)
2026-07-06 19:04:54.104 | ERROR    | kfactory.kcell:dup:781 - When duplicating a Cell, avoid giving the duplicate the same name, as this can cause naming conflicts and may render the GDS/OASIS file unwritable. If you're using a @cell function, ensure that the function has a different name than the one being called.
[via_MZI] splitter.o3 → via_fwd.o1 route length: 102679.712 µm
[via_MZI] via_rev.o1 → combiner.o1 route length: 103145.712 µm


In [ ]:
ILLEGAL = re.compile(r"[^A-Za-z0-9_$?.]")     # GDS-safe character set
DUP  = re.compile(r"^(?P<base>.+)\$\d+$")  # KLayout conflict suffix "$N"
BB    = re.compile(r"^AMF_")                # foundry black-box cells ONLY

def clean_cell_names(layout: kdb.Layout) -> None:
    # 1) Collapse "<bb>$N" -> exact canonical BB name, for BLACK-BOX cells only.
    #    User cells (bends, spirals, Unnamed_*) are left alone: their $N copies
    #    can be genuinely different cells, and merging them displaces geometry.
    changed = True
    while changed:
        changed = False
        for cell in list(layout.each_cell()):
            m = DUP.match(cell.name)
            if not m:
                continue
            base = m.group("base")
            if not BB.match(base):
                continue              # skip non-BB cells
            if layout.has_cell(base) and layout.cell_by_name(base) != cell.cell_index():
                canon = layout.cell(layout.cell_by_name(base))
                if cell.dbbox() != canon.dbbox():
                    continue                              # not identical -> don't merge
                for caller_idx in list(cell.caller_cells()):
                    caller = layout.cell(caller_idx)
                    for inst in list(caller.each_inst()):
                        if inst.cell_index == cell.cell_index():
                            ci = inst.cell_inst           # full CellInstArray (keeps transform)
                            ci.cell_index = canon.cell_index()
                            caller.insert(ci)
                            caller.erase(inst)
                layout.delete_cell(cell.cell_index())
                changed = True
                break
            elif not layout.has_cell(base):
                cell.name = base                          # only suffixed copy -> restore bb name
                changed = True
                break

    # 2) Strip illegal characters ("<defunct>$2" -> "defunct$2"); "$" is legal.
    seen = set(cl.name for cl in layout.each_cell())
    for cell in layout.each_cell():
        if ILLEGAL.search(cell.name):
            new = ILLEGAL.sub("", cell.name) or "cell"
            while new in seen and new != cell.name:
                new += "_x"
            seen.discard(cell.name); seen.add(new)
            cell.name = new

In [4]:
def keep_single_top(layout: kdb.Layout, top_name: str) -> None:
    """Make `top_name` the sole top cell; prune orphan tops (e.g. 'TOP' from
    imported sub-GDS) and any cells only they used. Shared BBs are kept."""
    if not layout.has_cell(top_name):
        raise ValueError(f"top cell {top_name!r} not found. tops present: "
                         f"{[c.name for c in layout.top_cells()]}")
    keep = layout.cell_by_name(top_name)
    removed = []
    for tc in list(layout.top_cells()):
        if tc.cell_index() != keep:
            removed.append(tc.name)
            tc.prune_cell()
    print("removed orphan top cells:", removed if removed else "none")

def drop_empty_cells(layout):
    removed = []
    changed = True
    while changed:                       # loop: deleting one can orphan another
        changed = False
        for cell in list(layout.each_cell()):
            if cell.is_top():
                continue                 # never delete the real top
            if cell.bbox().empty() and cell.child_instances() == 0:
                removed.append(cell.name)
                layout.delete_cell(cell.cell_index())
                changed = True
        # break out of for via while re-scan
    print("removed empty cells:", removed if removed else "none")

In [ ]:
# ---- Export gds ----
TMP   = "/workspace/myamf/gds/_tmp_main.gds"
FINAL = "/workspace/myamf/gds/Fonex_v6.gds"

# Stage 1: write WITHOUT metadata -> removes gdsfactory port/pin layers from bb cells
c.write_gds(TMP, with_metadata=False)

# Stage 2: clean cell names, then write final with polygons fractured to <= 4000 verts
layout = kdb.Layout()
layout.read(TMP)
keep_single_top(layout, "Main_v2")       


clean_cell_names(layout)
drop_empty_cells(layout)
for lnum in [(255, 0), (997, 0)]:
    li = layout.find_layer(*lnum)
    if li is not None:
        layout.clear_layer(li)     # removes the 274 pin-label shapes
        layout.delete_layer(li)    # removes the now-empty layer


opts = kdb.SaveLayoutOptions()
opts.gds2_max_vertex_count = 3000
opts.gds2_multi_xy_records = False
layout.write(FINAL, opts)
print("Wrote:", FINAL)



2668, record number=11293, cell=bend_euler_R891_A90_P0p5_W3_CSxs_5de03031_AMRVFalse)
removed empty cells: none
Wrote: /workspace/myamf/gds/Fonex_v6.gds


In [6]:
# ---- Verify the submission file ----
chk = kdb.Layout()
chk.read(FINAL)

bb_suffixed = [c.name for c in chk.each_cell()
               if BB.match(c.name) and re.search(r"\$\d+$", c.name)]   # BB names must be exact
illegal     = [c.name for c in chk.each_cell() if ILLEGAL.search(c.name)]  # banned on any cell
user_dollar = [c.name for c in chk.each_cell()
               if re.search(r"\$\d+$", c.name) and not BB.match(c.name)]   # allowed: user cells

worst = 0
for cell in chk.each_cell():
    for li in chk.layer_indexes():
        for s in cell.shapes(li).each():
            p = s.polygon
            if p:
                worst = max(worst, p.num_points_hull())

print("BB cells still suffixed   :", bb_suffixed if bb_suffixed else "none")
print("cells with illegal chars  :", illegal if illegal else "none")
print("user cells with $N (OK)   :", len(user_dollar))
print("max polygon vertices      :", worst)

assert not bb_suffixed, "A black-box cell still has a $N suffix!"
assert not illegal,     "A cell still has illegal characters!"
assert worst <= 4000,   "Still have polygons over 4000 vertices!"
print("OK - BB names exact, no illegal chars, all polygons <= 4000 vertices. Safe to submit.")
print("genuinely empty cells:", [c.name for c in chk.each_cell()
      if c.bbox().empty() and c.child_instances() == 0] or "none")
print("top cells:", [c.name for c in chk.top_cells()])

c = chk.cell(chk.cell_by_name("ECL_SiN_V2026_1"))
print("ECL_SiN_V2026_1 -> empty?", c.bbox().empty(),
      "| child_instances:", c.child_instances(), "| is_top:", c.is_top())
      
import klayout.db as kdb
ly = kdb.Layout(); ly.read("/workspace/myamf/gds/Fonex_v6.gds")
for lnum in [(255,0),(997,0)]:
    li = ly.find_layer(*lnum)
    if li is None:
        print(lnum, "-> not present"); continue
    tot = sum(c.shapes(li).size() for c in ly.each_cell())
    print(lnum, "->", tot, "shapes")

ly = kdb.Layout(); ly.read("/workspace/myamf/gds/Fonex_v6.gds")   # freshly re-exported
li = ly.find_layer(81,0)
print("81/0:", "gone" if li is None else f"{sum(c.shapes(li).size() for c in ly.each_cell())} shapes")
print("82/0:", ly.find_layer(82,0) is not None)



BB cells still suffixed   : none
cells with illegal chars  : none
user cells with $N (OK)   : 48
max polygon vertices      : 2926
OK - BB names exact, no illegal chars, all polygons <= 4000 vertices. Safe to submit.
genuinely empty cells: none
top cells: ['Main_v2']
ECL_SiN_V2026_1 -> empty? False | child_instances: 34 | is_top: False
(255, 0) -> not present
(997, 0) -> not present
81/0: gone
82/0: True
